In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# -------- Data --------
eng = ["i am good", "you are good", "i am bad"]
fra = [" je suis bon",
       "tu es bon",
       "je suis mauvais"]

# -------- Tokenization --------
tok_eng = Tokenizer()
tok_fra = Tokenizer()

tok_eng.fit_on_texts(eng)
tok_fra.fit_on_texts(fra)

X = pad_sequences(tok_eng.texts_to_sequences(eng))
y_seq = tok_fra.texts_to_sequences(fra)

y = pad_sequences(y_seq, maxlen=X.shape[1])

# -------- Model --------
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Embedding(len(tok_eng.word_index)+1, 8),
    tf.keras.layers.SimpleRNN(16, return_sequences=True),
    tf.keras.layers.Dense(len(tok_fra.word_index)+1, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# -------- Train --------
model.fit(X, y, epochs=200, verbose=0)

# -------- Test --------
test_sentence = "I am good"
test = pad_sequences(tok_eng.texts_to_sequences([test_sentence]), maxlen=X.shape[1])

pred = model.predict(test)

# -------- Convert index → words --------
pred_ids = pred.argmax(axis=-1)[0]

words = []
for i in pred_ids:
    word = tok_fra.index_word.get(i, "")
    if word != "":
      words.append(word)

print("Input:", test_sentence)
print("Translation:", " ".join(words))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
Input: I am good
Translation: je suis bon
